In [ ]:
# ------- Google Drive -------
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Show versioning of deep learning libraries
import torch, torchvision
print(torch.__version__, torch.cuda.is_available())

In [ ]:
!pip install torchstain

**Import libraries**

In [4]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
from tqdm import tqdm
from skimage.measure import label
from skimage.morphology import remove_small_objects
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from scipy import ndimage as ndi

**Define paths**

In [ ]:
# Modifica queste righe nel notebook 'post'
checkpoint_dir = ""
my_mask = os.path.join(checkpoint_dir, "best_validation_masks")

# Assicurati che il percorso dei dati di test/ground truth sia corretto
# Se usi il K-Fold, punta alla directory dei fold per trovare le maschere manuali
test_dir = ""
# Nota: dovrai poi specificare il fold corretto (es. fold_01) o una cartella aggregata
TEST_MASKS_DIR = os.path.join(test_dir, "fold_05/manual")

**U-Net**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    """Due strati consecutivi di conv-batchnorm-relu"""
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, init_filters=64, depth=4, bilinear=True):
        super(UNet, self).__init__()
        self.depth = depth
        self.down_layers = nn.ModuleList()
        self.up_layers = nn.ModuleList()
        self.pool = nn.MaxPool2d(2)

        # Encoder
        filters = init_filters
        for d in range(depth):
            conv = DoubleConv(in_channels, filters)
            self.down_layers.append(conv)
            in_channels = filters
            filters *= 2

        # Bottleneck
        self.bottleneck = DoubleConv(in_channels, filters)

        # Decoder
        for d in range(depth):
            filters //= 2
            if bilinear:
                up = nn.Sequential(
                    nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
                    nn.Conv2d(filters * 2, filters, kernel_size=1)
                )
            else:
                up = nn.ConvTranspose2d(filters * 2, filters, kernel_size=2, stride=2)

            conv = DoubleConv(filters * 2, filters)
            # Utilizzo di ModuleDict per mantenere la compatibilità con i pesi salvati
            self.up_layers.append(nn.ModuleDict({'up': up, 'conv': conv}))

        # Final layer - RINOMINATO per caricare correttamente lo state_dict
        self.final_conv = nn.Conv2d(init_filters, out_channels, kernel_size=1)

    def forward(self, x):
        # Encoder con skip connections
        skips = []
        for down in self.down_layers:
            x = down(x)
            skips.append(x)
            x = self.pool(x)

        # Bottleneck
        x = self.bottleneck(x)

        # Decoder con skip connections (ordine inverso)
        skips = skips[::-1]
        for i, up_layer in enumerate(self.up_layers):
            x = up_layer['up'](x)
            skip = skips[i]

            # Gestione eventuali mismatch di dimensione (es. input dispari)
            if x.shape != skip.shape:
                diff_h = skip.shape[2] - x.shape[2]
                diff_w = skip.shape[3] - x.shape[3]
                x = F.pad(x, [diff_w // 2, diff_w - diff_w // 2,
                             diff_h // 2, diff_h - diff_h // 2])

            x = torch.cat([skip, x], dim=1)
            x = up_layer['conv'](x)

        # Ritorna il risultato del layer final_conv
        return self.final_conv(x)

In [ ]:
# Model hyperparameters - Allineati con model_prep_HED
in_channels = 3  # RGB
out_channels = 1
init_filters = 64
depth = 4
IMG_SIZE = (416, 416)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Model configuration:")
print(f"  in_channels: {in_channels}")
print(f"  out_channels: {out_channels}")
print(f"  init_filters: {init_filters}")
print(f"  depth: {depth}")

# Initialize model (Aggiunto bilinear=True per matchare il training)
model = UNet(in_channels=in_channels, out_channels=out_channels,
             init_filters=init_filters, depth=depth, bilinear=True).to(DEVICE)

# Load best model checkpoint
best_model_path = os.path.join(checkpoint_dir, "best_model.pth")
if not os.path.exists(best_model_path):
    raise FileNotFoundError(f"Best model not found: {best_model_path}")

print(f"\nLoading checkpoint: {best_model_path}")
checkpoint = torch.load(best_model_path, map_location=DEVICE, weights_only=False)

# Load model weights
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Show checkpoint info - Aggiornato per riflettere le chiavi salvate in HED
epoch = checkpoint.get('epoch', 'N/A')
best_dsc_strict = checkpoint.get('dsc_strict', None) # HED usa 'dsc_strict'
val_fold = checkpoint.get('val_fold', 'N/A') # HED salva anche il fold di validazione

print(f"\nModel loaded successfully!")
print(f"Checkpoint info:")
print(f"  Epoch: {int(epoch) + 1 if isinstance(epoch, int) else epoch}")
print(f"  Validation Fold: {val_fold}")

**Over/Under segmentation**

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from skimage.measure import label
from matplotlib.patches import Patch
from matplotlib.gridspec import GridSpec

# --- 1. CONFIGURAZIONE PERCORSI ---
checkpoint_dir = ""
PREDICTED_DIR = os.path.join()
GT_DIR = os.path.join("")
images_dir = os.path.join("")
FINAL_SIZE = (416, 416)

def diagnose_target_density(target_n=40):
    """
    Cerca immagini con circa 'target_n' vacuoli (es. 40).
    Risolve il problema di sovrapposizione delle label.
    """
    print(f"🔬 RICERCA CASI: Target specifico (~{target_n} oggetti)")
    print(f"{'='*70}\n")

    pred_files = [f for f in os.listdir(PREDICTED_DIR) if f.endswith('.png')]
    all_results = []

    # --- FASE 1: RACCOLTA DATI ---
    print(f"📊 Ricerca immagini con circa {target_n} steatosi...")
    for filename in tqdm(pred_files, desc="Scanning", ncols=80):
        p_img = cv2.imread(os.path.join(PREDICTED_DIR, filename), cv2.IMREAD_GRAYSCALE)
        gt_f = filename.replace("_pred", "")
        g_img = cv2.imread(os.path.join(GT_DIR, gt_f), cv2.IMREAD_GRAYSCALE)

        if p_img is None or g_img is None:
            continue

        p_bin = (cv2.resize(p_img, FINAL_SIZE, interpolation=cv2.INTER_NEAREST) > 0).astype(np.uint8)
        g_bin = (cv2.resize(g_img, FINAL_SIZE, interpolation=cv2.INTER_NEAREST) > 0).astype(np.uint8)

        _, n_g = label(g_bin, return_num=True) # Conta oggetti reali
        _, n_p = label(p_bin, return_num=True)
        err_cnt = n_p - n_g

        all_results.append({
            'fname': filename, 'gt_f': gt_f,
            'p_bin': p_bin, 'g_bin': g_bin,
            'n_g': n_g,
            'n_p': n_p,
            'err_cnt': err_cnt
        })

    # --- FASE 2: FILTRAGGIO PER TARGET (40) ---
    if not all_results:
        print("Nessuna immagine trovata."); return

    # Ordina in base alla differenza assoluta dal target (chi è più vicino a 40 vince)
    all_results.sort(key=lambda x: abs(x['n_g'] - target_n))

    representative_cases = all_results[:2]

    print(f"\n Trovati i casi più vicini a {target_n}:")
    print(f"   Case 1 Count: {representative_cases[0]['n_g']} vacuoli")
    if len(representative_cases) > 1:
        print(f"   Case 2 Count: {representative_cases[1]['n_g']} vacuoli")

    # --- FASE 3: VISUALIZZAZIONE ---
    COLOR_TP = np.array([0, 255, 0])   # Verde (Corretto)
    COLOR_FP = np.array([255, 255, 0]) # Giallo (Sovrastima)
    COLOR_FN = np.array([255, 0, 0])   # Rosso (Perso)

    # Aumento leggermente l'altezza della figura per dare spazio alle label
    fig = plt.figure(figsize=(22, 12))

    # Regolo i margini: 'top=0.85' lascia molto spazio in alto per titolo e legenda
    gs = GridSpec(2, 4, figure=fig, hspace=0.25, wspace=0.15,
                  top=0.85, bottom=0.05, left=0.05, right=0.95)

    for i, case in enumerate(representative_cases):
        rgb = cv2.imread(os.path.join(images_dir, case['gt_f']))
        if rgb is not None:
            rgb = cv2.cvtColor(cv2.resize(rgb, FINAL_SIZE), cv2.COLOR_BGR2RGB)
        else:
            rgb = np.zeros(FINAL_SIZE + (3,), dtype=np.uint8)

        overlay = np.zeros(FINAL_SIZE + (3,), dtype=np.uint8)
        mask_tp = (case['p_bin'] == 1) & (case['g_bin'] == 1)
        mask_fp = (case['p_bin'] == 1) & (case['g_bin'] == 0)
        mask_fn = (case['p_bin'] == 0) & (case['g_bin'] == 1)

        overlay[mask_tp] = COLOR_TP
        overlay[mask_fp] = COLOR_FP
        overlay[mask_fn] = COLOR_FN

        axes = [fig.add_subplot(gs[i, j]) for j in range(4)]

        axes[0].imshow(rgb); axes[0].set_title('Original', fontsize=12)
        axes[1].imshow(case['g_bin'], cmap='gray'); axes[1].set_title(f"GT ({case['n_g']} objs)", fontsize=12)
        axes[2].imshow(case['p_bin'], cmap='gray'); axes[2].set_title(f"Pred ({case['n_p']} objs)", fontsize=12)
        axes[3].imshow(overlay); axes[3].set_title('Segmentation Overlay', fontsize=12)

        # Info errore
        axes[3].text(0.5, -0.15, f"Diff: {case['err_cnt']:+d} objects",
                     transform=axes[3].transAxes, ha='center', fontsize=11, fontweight='bold')

        for ax in axes: ax.axis('off')

    # LEGENDA E TITOLO (Posizionamento corretto)
    legend_elements = [
        Patch(facecolor=COLOR_TP/255, edgecolor='black', label='True Positive (Green)'),
        Patch(facecolor=COLOR_FP/255, edgecolor='black', label='False Positive (Yellow)'),
        Patch(facecolor=COLOR_FN/255, edgecolor='black', label='False Negative (Red)')
    ]

    # Metto la legenda ben separata dal titolo
    # bbox_to_anchor=(0.5, 0.92) la posiziona appena sotto il titolo
    fig.legend(handles=legend_elements, loc='upper center', ncol=3, fontsize=13,
               bbox_to_anchor=(0.5, 0.92), frameon=True, edgecolor='black')

    # Titolo più in alto (y=0.98)
    plt.suptitle(f"Analysis: Specific Density Cases (~{target_n} objects)",
                 fontsize=18, fontweight='bold', y=0.98)

    plt.show()

# Esecuzione per ~40 oggetti
diagnose_target_density(target_n=40)

**Post-processing functions**

In [ ]:
def apply_clean(mask, min_size=50):
    """Rimuove piccoli oggetti (rumore/falsi positivi)."""
    cleaned = remove_small_objects((mask > 0).astype(bool), min_size=min_size)
    return (cleaned.astype(np.uint8) * 255)

def apply_morph(mask, kernel_size=3):
    """Opening + Closing per pulire contorni e buchi interni."""
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    m = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    return cv2.morphologyEx(m, cv2.MORPH_CLOSE, kernel)

def apply_watershed(mask):
    """Separa vacuoli uniti usando la Distance Transform."""
    if mask.max() == 0: return mask
    dist = ndi.distance_transform_edt(mask > 0)
    dist_s = ndi.gaussian_filter(dist, sigma=0.8)
    coords = peak_local_max(dist_s, min_distance=7, labels=(mask > 0))
    if len(coords) == 0: return mask
    peaks = np.zeros(dist.shape, dtype=bool); peaks[tuple(coords.T)] = True
    markers, _ = label(peaks, return_num=True)
    return (watershed(-dist_s, markers, mask=(mask > 0)) > 0).astype(np.uint8) * 255

def adaptive_postprocess(pred, gt):
    """Applica il post-processing solo se migliora il DSC rispetto alla baseline."""
    p_b, g_b = (pred > 0).astype(float), (gt > 0).astype(float)
    dsc_base = (2. * np.sum(p_b * g_b)) / (np.sum(p_b) + np.sum(g_b) + 1e-6)

    if dsc_base > 0.85: return apply_clean(pred, min_size=20), 'clean_light'

    candidate = apply_clean(apply_watershed(pred), min_size=40)
    cand_b = (candidate > 0).astype(float)
    dsc_new = (2. * np.sum(cand_b * g_b)) / (np.sum(cand_b) + np.sum(g_b) + 1e-6)

    return (candidate, 'watershed_clean') if dsc_new > dsc_base else (pred, 'baseline')

def apply_refined_post_processing(mask, threshold=127, min_size=50, kernel_size=3):
    """
    1. Binarizzazione
    2. Chiusura morfologica
    3. Hole Filling
    4. Area Filtering
    """
    # 1. Binarizzazione
    if mask.dtype != np.uint8:
        mask = (mask * 255).astype(np.uint8) if mask.max() <= 1.0 else mask.astype(np.uint8)
    _, binary = cv2.threshold(mask, threshold, 255, cv2.THRESH_BINARY)

    # 2. Chiusura Morfologica (Kernel 3x3)
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)

    # 3. Hole Filling (Riempimento buchi interni)
    contours, _ = cv2.findContours(closed, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in contours:
        cv2.drawContours(closed, [cnt], 0, 255, -1)

    # 4. Area Filtering (Rimozione rumore)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(closed, connectivity=8)
    final_mask = np.zeros_like(closed)
    for i in range(1, num_labels):
        if stats[i, cv2.CC_STAT_AREA] >= min_size:
            final_mask[labels == i] = 255

    return final_mask

def apply_adaptive_density_based(mask, density_threshold=0.001):
    """
    Post-processing adattivo basato sulla densità di foreground.

    Strategia (da best_postprocessing.py):
    - Se densità < 0.001: Applica Close3 + Clean44
    - Altrimenti: Raw (nessun processing)

    """
    # Normalizza a 0-1
    if mask.max() > 1:
        mask_norm = mask.astype(np.float32) / 255.0
    else:
        mask_norm = mask.astype(np.float32)

    # Calcola densità foreground
    binary = (mask_norm > 0.5).astype(np.float32)
    fg_density = binary.sum() / binary.size

    if fg_density < density_threshold:
        # BASSA DENSITÀ
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        closed = cv2.morphologyEx((mask_norm * 255).astype(np.uint8),
                                   cv2.MORPH_CLOSE, kernel)

        # Remove Small Objects (min_size=44)
        binary_closed = (closed > 127).astype(np.uint8)
        labeled = label(binary_closed)

        for region_id in range(1, labeled.max() + 1):
            region_mask = (labeled == region_id)
            if region_mask.sum() < 44:
                binary_closed[region_mask] = 0

        result = binary_closed * 255
    else:
        # ALTA DENSITÀ
        result = (binary * 255).astype(np.uint8)

    return result

def apply_closing(mask, kernel_size=3):
    """
    Morphological Closing (Dilation → Erosion)
    - Chiude piccoli buchi interni
    - Connette regioni vicine
    """
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    return cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)


def apply_opening(mask, kernel_size=3):
    """
    Morphological Opening (Erosion → Dilation)
    - Rimuove piccole protuberanze
    - Elimina rumore esterno
    """
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    return cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)


def apply_closing_then_opening(mask, kernel_size=3):
    """
    Closing → Opening
    - Prima chiude i buchi (Closing)
    - Poi rimuove il rumore (Opening)
    - Utile per: maschere con buchi + rumore esterno
    """
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    closed = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    return cv2.morphologyEx(closed, cv2.MORPH_OPEN, kernel)


def apply_opening_then_closing(mask, kernel_size=3):
    """
    Opening → Closing
    - Prima rimuove il rumore (Opening)
    - Poi chiude i buchi (Closing)
    - Utile per: maschere rumorose con pochi buchi
    - Questa è la funzione apply_morph() originale!
    """
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    opened = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    return cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel)

In [ ]:
def calculate_dice_score_single(pred_bin, target):
    """Calcola DSC per una SINGOLA immagine (già binarizzata)."""
    pred_flat = pred_bin.view(-1)
    target_flat = target.view(-1)
    intersection = (pred_flat * target_flat).sum().item()
    pred_sum = pred_flat.sum().item()
    target_sum = target_flat.sum().item()
    union = pred_sum + target_sum
    smooth = 1e-5

    dsc_standard = (2. * intersection + smooth) / (union + smooth) if union != 0 else 1.0

    if target_sum == 0:
        dsc_strict = float('nan')
    elif union == 0:
        dsc_strict = float('nan')
    else:
        dsc_strict = (2. * intersection) / union
    return {'dsc_standard': dsc_standard, 'dsc_strict': dsc_strict}

def calculate_dice_score(pred, target, threshold=0.5):
    """Calcola DSC per batch o singola immagine (PyTorch)."""
    if not isinstance(pred, torch.Tensor): pred = torch.from_numpy(pred).float()
    if not isinstance(target, torch.Tensor): target = torch.from_numpy(target).float()

    # --- FIX CRUCIALE ---
    # Se i dati sono in 0-255 (PNG caricate), normalizziamo a 0-1 PRIMA del controllo sigmoid.
    # Questo evita che sigmoid(0) diventi 0.5, alterando il DSC.
    if pred.max() > 1.0: pred /= 255.0
    if target.max() > 1.0: target /= 255.0

    if pred.min() < 0 or pred.max() > 1:
        pred = torch.sigmoid(pred)

    pred_bin = (pred > threshold).float()

    # Gestione dimensioni per batch o singola immagine
    if pred_bin.dim() == 2: pred_bin = pred_bin.unsqueeze(0).unsqueeze(0)
    elif pred_bin.dim() == 3: pred_bin = pred_bin.unsqueeze(0)
    if target.dim() == 2: target = target.unsqueeze(0).unsqueeze(0)
    elif target.dim() == 3: target = target.unsqueeze(0)

    batch_size = pred_bin.size(0)
    dsc_std_list, dsc_strict_list = [], []
    for i in range(batch_size):
        scores = calculate_dice_score_single(pred_bin[i], target[i])
        dsc_std_list.append(scores['dsc_standard'])
        dsc_strict_list.append(scores['dsc_strict'])
    return {'dsc_standard_list': dsc_std_list, 'dsc_strict_list': dsc_strict_list}


def calculate_metrics(pred_mask, gt_mask, threshold=0.5):
    """Calcola le metriche forzando i float per eliminare i valori RVD errati."""
    # 1. DSC (Logica training)
    res = calculate_dice_score(pred_mask, gt_mask, threshold=threshold)
    dsc_std = res['dsc_standard_list'][0]
    dsc_strict = res['dsc_strict_list'][0]

    # 2. Conversione in binario 0-1 (NumPy)
    p = (pred_mask > 127).astype(np.uint8) if pred_mask.max() > 1 else (pred_mask > threshold).astype(np.uint8)
    g = (gt_mask > 127).astype(np.uint8) if gt_mask.max() > 1 else (gt_mask > 0).astype(np.uint8)

    # 3. Metriche morfologiche
    intersection = np.logical_and(p, g).sum()
    iou = intersection / (np.logical_or(p, g).sum() + 1e-6)
    _, n_p = label(p, return_num=True)
    _, n_g = label(g, return_num=True)

    # --- FIX RVD: Forziamo il calcolo in FLOAT prima della sottrazione ---
    area_p = float(np.sum(p)) #
    area_g = float(np.sum(g)) #

    sp_err = abs((area_p / p.size) - (area_g / g.size)) * 100

    # Relative Volume Difference corretta: (Pred - GT) / GT
    rvd = ((area_p - area_g) / (area_g + 1e-6)) * 100

    return dsc_std, dsc_strict, iou, abs(n_p - n_g), sp_err, rvd

In [ ]:
import pandas as pd
def run_pipeline():
    # Sincronizza percorsi
    predicted_masks_dir = my_mask
    manual_dir = TEST_MASKS_DIR
    pred_files = [f for f in os.listdir(predicted_masks_dir) if f.endswith('.png')]

    strategies = {
        "1. Baseline": lambda m, g: (m, None),
        "2. Clean (45px)": lambda m, g: (apply_clean(m, min_size=45), None),
        "3. Morph (O→C k=3)": lambda m, g: (apply_morph(m, kernel_size=3), None),
        "4. Adaptive": adaptive_postprocess,
        "5. Closing Only": lambda m, g: (apply_closing(m, kernel_size=3), None),
        "6. Opening Only": lambda m, g: (apply_opening(m, kernel_size=3), None),
        "7. Close→Open": lambda m, g: (apply_closing_then_opening(m, kernel_size=3), None),
        "8. Open→Close": lambda m, g: (apply_opening_then_closing(m, kernel_size=3), None),
        "9. Watershed+Clean": lambda m, g: (apply_clean(apply_watershed(m), min_size=16), None),
        "10. Full (W+M+C)": lambda m, g: (apply_clean(apply_morph(apply_watershed(m)), min_size=45), None),
        "11. Refined (M+H+C)": lambda m, g: (apply_refined_post_processing(m, threshold=100, min_size=45, kernel_size=3), None),
        "12. Density-Based": lambda m, g: (apply_adaptive_density_based(m, density_threshold=0.001), None),
        "13. Closing+Clean45": lambda m, g: (apply_clean(apply_closing(m, kernel_size=3), min_size=45), None),
        "14. Opening+Clean45": lambda m, g: (apply_clean(apply_opening(m, kernel_size=3), min_size=45), None),
        "15. C→O+Clean45": lambda m, g: (apply_clean(apply_closing_then_opening(m, kernel_size=3), min_size=45), None)
    }

    all_res = {name: {k: [] for k in ['std', 'strict', 'iou', 'cnt', 'area', 'rvd']} for name in strategies.keys()}

    for filename in tqdm(pred_files, desc="Valutazione"):
        p_orig = cv2.imread(os.path.join(predicted_masks_dir, filename), 0)
        gt_f = filename.replace("_pred", "")
        gt_path = os.path.join(manual_dir, gt_f)
        if not os.path.exists(gt_path):
            gt_path = os.path.join(manual_dir, gt_f.replace(".png", "_mask.png"))

        g_orig = cv2.imread(gt_path, 0)
        if p_orig is None or g_orig is None:
            continue

        p_orig = cv2.resize(p_orig, (416, 416), interpolation=cv2.INTER_NEAREST)
        g_orig = cv2.resize(g_orig, (416, 416), interpolation=cv2.INTER_NEAREST)

        for name, func in strategies.items():
            proc, _ = func(p_orig, g_orig)
            std, strict, iou, cnt, area, rvd = calculate_metrics(proc, g_orig)

            all_res[name]['std'].append(std)
            all_res[name]['strict'].append(strict)
            all_res[name]['iou'].append(iou)
            all_res[name]['cnt'].append(cnt)
            all_res[name]['area'].append(area)
            # Filtra RVD outlier
            if abs(rvd) < 1000:
                all_res[name]['rvd'].append(rvd)

    # Report finale (ordine originale)
    report = []
    for name in strategies.keys():
        report.append({
            'Strategy': name,
            'DSC Std': f"{np.mean(all_res[name]['std']):.4f}",
            'DSC Strict': f"{np.nanmean(all_res[name]['strict']):.4f}",
            'RVD %': f"{np.mean(all_res[name]['rvd']):.2f}%",
            'IoU': f"{np.mean(all_res[name]['iou']):.4f}",
            'MAE Count': f"{np.mean(all_res[name]['cnt']):.2f}",
            'SP Error %': f"{np.mean(all_res[name]['area']):.3f}%"
        })

    print("\n" + "="*120)
    print("REPORT CONFRONTO POST-PROCESSING ESTESO (15 STRATEGIE)")
    print("="*120)
    print(pd.DataFrame(report).to_string(index=False))
    print("="*120)

    # Trova il miglior DSC senza ordinare la tabella
    best_dsc = max([np.mean(all_res[name]['std']) for name in strategies.keys()])
    best_strategy = [name for name in strategies.keys() if np.mean(all_res[name]['std']) == best_dsc][0]

    print(f"\n MIGLIOR STRATEGIA: {best_strategy} (DSC: {best_dsc:.4f})")
    print()

# Esegui
run_pipeline()